# ChakraTransformer Kaggle Evaluation

This notebook evaluates the pre-trained ChakraTransformer (Combo 6) on the Kvasir-SEG test set.

In [1]:
# !pip install -q timm opencv-python
# !cp -r /kaggle/input/chakramodel-data/src /kaggle/working/
# import sys
# sys.path.append('/kaggle/working/src')


In [ ]:
# Cell 1: Copying source files
!pip install -q timm opencv-python
!cp -r /kaggle/input/updated_kaggle/src /kaggle/working/
import sys
sys.path.append('/kaggle/working/src')

In [ ]:
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from chakra_transformer.transformer_segmenter import ChakraTransformerSegmenter
from train_pranet import KvasirSEGDataset
from evaluate_all import evaluate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# Dataset Setup
images_dir = Path('/kaggle/input/kvasir-seg/Kvasir-SEG/images')
masks_dir = Path('/kaggle/input/kvasir-seg/Kvasir-SEG/masks')

# 448x448 is required for vit_large_patch16_384 (or 384x384 if configured for it)
dataset = KvasirSEGDataset(images_dir, masks_dir, img_size=448, augment=False)
all_indices = list(range(len(dataset)))

# Standard Split: 70% train | 15% calib | 15% test (or use 10% test depending on earlier paper)
# We will use 10% test to match the paper benchmark.
n_train = int(0.7 * len(dataset))
n_cal = int(0.15 * len(dataset))
test_indices = all_indices[n_train + n_cal:]
test_set = torch.utils.data.Subset(dataset, test_indices)
test_loader = DataLoader(test_set, batch_size=8, shuffle=False, num_workers=2)
print(f"Test set size: {len(test_set)}")


In [ ]:
# # Load Model
# print("Loading ChakraTransformer...")
# model = ChakraTransformerSegmenter(backbone_name='vit_large_patch16_384', pretrained=False)
# weight_path = '/kaggle/input/chakramodel-data/weights/chakra_transformer_best.pth'
# model.load_state_dict(torch.load(weight_path, map_location=device))
# model.to(device).eval()
# print("Model loaded successfully.")


In [ ]:
# Cell 4: Loading model weights
print("Loading ChakraTransformer...")
model = ChakraTransformerSegmenter(backbone_name='vit_large_patch16_384', pretrained=False)
weight_path = '/kaggle/input/updated_kaggle/chakra_transformer_best.pth'
model.load_state_dict(torch.load(weight_path, map_location=device))

In [ ]:
# Evaluate
print("Running Evaluation...")
metrics = evaluate_model(model, test_loader, device)
print("\n=== FINAL RESULTS ===")
print(f"Dice Score: {metrics['dice']:.4f}")
print(f"mIoU Score: {metrics['iou']:.4f}")
print(f"Precision:  {metrics['precision']:.4f}")
print(f"Recall:     {metrics['recall']:.4f}")


## Video Inference on polypdataset-gokul

Run the ChakraTransformer on all `.mp4` and `.avi` videos from the dataset and save the annotated videos.

In [ ]:
import cv2
import numpy as np
from torchvision import transforms
import time

video_dir = Path('/kaggle/input/polypdataset-gokul')
output_dir = Path('/kaggle/working/output_videos')
output_dir.mkdir(parents=True, exist_ok=True)

video_files = list(video_dir.glob('*.mp4')) + list(video_dir.glob('*.avi'))
print(f"Found {len(video_files)} videos for testing.")

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])

for vid_path in video_files:
    print(f"\nProcessing {vid_path.name}...")
    cap = cv2.VideoCapture(str(vid_path))
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    
    out_path = str(output_dir / f"annotated_{vid_path.name}")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, int(fps), (width, height))
    
    frame_count = 0
    start_time = time.time()
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        input_tensor = transform(rgb_frame).unsqueeze(0).to(device)
        
        with torch.no_grad():
            outputs = model(input_tensor)
            if isinstance(outputs, (tuple, list)):
                pred = outputs[0]
            else:
                pred = outputs
            pred = torch.sigmoid(pred).squeeze().cpu().numpy()
            
        pred_resized = cv2.resize(pred, (width, height))
        mask = (pred_resized > 0.5).astype(np.uint8) * 255
        
        colored_mask = np.zeros_like(frame)
        colored_mask[:, :, 1] = mask
        
        overlay = cv2.addWeighted(frame, 0.7, colored_mask, 0.3, 0)
        out.write(overlay)
        
        frame_count += 1
        if frame_count % 50 == 0:
            print(f"  Processed {frame_count} frames...")
            
    cap.release()
    out.release()
    elapsed = time.time() - start_time
    print(f"Finished {vid_path.name} - {frame_count} frames in {elapsed:.2f}s ({frame_count/elapsed:.1f} FPS)")
    print(f"Saved to {out_path}")
